[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/smolagents/smolagents_orchestrator.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

## SmolAgents 编排器与工作智能体教程

本教程介绍如何使用 SmolAgents 实现编排器与工作智能体模式，并使用 Phoenix 追踪工作流。

该模式由中央编排器协调多个负责具体任务的工作智能体，有助于提高工作流的模块化程度与扩展能力。追踪可以帮助我们观察协调过程，了解编排器与工作智能体的交互。

SmolAgents 的灵活智能体能力可以与 Python 的 asyncio 库结合，编排多个智能体之间的交互。

完成本教程后，你将学会：

- 为编排器与工作智能体配置合适的 SmolAgents 工具
- 使用 Python asyncio 库实现这一模式
- 使用 Phoenix 追踪并可视化智能体之间的交互
- 比较编排工作流与非编排工作流的性能

⚠️ 本教程需要 Hugging Face 令牌。

## 设置密钥与依赖

In [ ]:
!pip install openinference-instrumentation-smolagents smolagents

In [ ]:
import os
from getpass import getpass

from phoenix.otel import register
from smolagents import CodeAgent, HfApiModel, tool

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("🔑 请输入 Hugging Face 令牌： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置模型

In [ ]:
# 初始化模型
model = HfApiModel()

## 配置追踪

In [ ]:
tracer_provider = register(
    project_name="smolagents-agents",
    protocol="http/protobuf",
    auto_instrument=True,
)

## 定义工具

In [ ]:
@tool
def extract_qualifications(profile: str) -> str:
    """
    从候选人简介中提取资历。

    Args:
        profile: 候选人背景或简历摘要。

    Returns:
        关键技能、经历或教育背景列表。
    """
    keywords = [word.strip(".,") for word in profile.split() if len(word) > 5]
    return ", ".join(set(keywords))


@tool
def analyze_tone(profile: str) -> str:
    """
    根据写作风格分析语气或软技能。

    Args:
        profile: 候选人的自我描述。

    Returns:
        沟通风格与语气观察结果。
    """
    return "Confident, technically focused, concise communicator."

## 工作智能体

In [ ]:
def resume_screener_agent(profile: str, role: str) -> str:
    agent = CodeAgent(tools=[extract_qualifications], model=model)
    prompt = (
        f"请按以下职位评审这份简介： {role}:\n\n{profile}\n\n"
        f"使用 extract_qualifications 提取技能，再根据相关性给出简短评分（1–10）。"
    )
    return agent.run(prompt)


def culture_fit_agent(profile: str) -> str:
    agent = CodeAgent(tools=[analyze_tone], model=model)
    prompt = (
        f"根据以下候选人自我描述：\n\n{profile}\n\n"
        f"使用 analyze_tone 评估沟通风格，然后评论其潜在团队或文化匹配程度。"
    )
    return agent.run(prompt)


def recommendation_agent(tech_eval: str, culture_eval: str) -> str:
    agent = CodeAgent(tools=[], model=model)
    prompt = (
        f"招聘人员收到以下评估：\n\n"
        f"技术评估：\n{tech_eval}\n\n"
        f"文化匹配评估：\n{culture_eval}\n\n"
        f"综合两项评估，该候选人是否应进入下一轮？给出 yes/no，并简要说明。"
    )
    return agent.run(prompt)

## 编排器

In [ ]:
def orchestrator(candidate_name: str, profile: str, role: str):
    print(f"\n🧑‍💼 正在评审： {candidate_name} — 职位： {role}")

    # 第 1 步：获取技术筛选结果
    tech_eval = resume_screener_agent(profile, role)

    # 第 2 步：获取文化匹配筛选结果
    culture_eval = culture_fit_agent(profile)

    # 第 3 步：获取最终建议
    final_rec = recommendation_agent(tech_eval, culture_eval)

    # 显示结果
    print(f"\n📄 技术评估：\n{tech_eval}")
    print(f"\n💬 文化匹配评估：\n{culture_eval}")
    print(f"\n✅ 推荐意见：\n{final_rec}")
    print("\n" + "=" * 60 + "\n")

## 并行运行多个候选任务

In [ ]:
def main():
    role = "Machine Learning Engineer"
    candidates = {
        "Sofia Rao": """5+ years in applied ML. Experience with Transformers, model evaluation, and production pipelines. Previously led a small team at a healthtech startup.""",
        "Daniel Kim": """Worked as a backend engineer, now transitioning into ML. Strong Python and infra background. No formal ML experience, but completed multiple online LLM projects.""",
    }

    for name, profile in candidates.items():
        orchestrator(name, profile, role)


if __name__ == "__main__":
    main()